# Staged mocks wedge: 3D cosmic web funnel (truth-only)

Interactive **Plotly** 3D scatter of Abacus SecondGen mock galaxies in a fixed survey wedge, colored by **T-Web** environment class (void / wall / filament / cluster).

## Four-panel funnel

| Stage | NPZ | Description |
|-------|-----|-------------|
| 1 | `staged_mock_wedge_truth_annotated_rs7.npz` | Full wedge cut from annotated CutSky (truth labels) |
| 2 | `staged_mock_wedge_stage1_rmaglt19p5_rs7.npz` | Stage 1: apparent magnitude cut `R_MAG_APP < 19.5` |
| 3 | `staged_mock_wedge_forFA_rs7.npz` | Stage 2: `forFA` target selection |
| 4 | `staged_mock_wedge_stage3_postcollision_rs7.npz` | Stage 3: science sample, `COLLISION==0`, unique halo triple |

**Wedge bounds** (documented in config): RA 120–140°, Dec 16.5–26.7°, redshift `Z` ∈ [0.25, 0.30].

**Classification**: count of eigenvalues above threshold — `(λ₁, λ₂, λ₃ > λ_thr).sum()` → 0=void, 1=wall, 2=filament, 3=cluster, with **`λ_thr = 0.2`**.

Positions are converted from **RA, DEC, Z** to comoving Cartesian **Mpc** using **Planck18** (not Mpc/h).

> **Inputs**: NPZs are produced by the staged-mock wedge pipeline under  
> `/pscratch/sd/d/dkololgi/abacus/SecondGen_Mocks/ph000/wedge/`.  
> This notebook does **not** use DESI inference outputs; run it after those NPZs exist.


In [ ]:
from pathlib import Path
import numpy as np

# --- NPZ paths (staged-mock wedge pipeline outputs) ---
WEDGE_DIR = Path("/pscratch/sd/d/dkololgi/abacus/SecondGen_Mocks/ph000/wedge").expanduser().resolve()

STAGES = [
    ("Truth (annotated CutSky wedge)", WEDGE_DIR / "staged_mock_wedge_truth_annotated_rs7.npz"),
    ("Stage 1 (R_MAG_APP < 19.5)", WEDGE_DIR / "staged_mock_wedge_stage1_rmaglt19p5_rs7.npz"),
    ("forFA targets", WEDGE_DIR / "staged_mock_wedge_forFA_rs7.npz"),
    ("Stage 3 (post-collision, unique)", WEDGE_DIR / "staged_mock_wedge_stage3_postcollision_rs7.npz"),
]

# Wedge bounds used when building the NPZs (for documentation / optional re-filter)
RA_MIN, RA_MAX = 120.0, 140.0
DEC_MIN, DEC_MAX = 16.5, 26.7
Z_MIN, Z_MAX = 0.25, 0.30

LAMBDA_THRESHOLD = 0.2

# Plot subsample (markers only); set None to plot all points
MAX_POINTS = 250_000
SEED = 0

# Units: comoving distance in Mpc (Planck18). Set True for Mpc/h.
MPC_H = False

# HTML output (same directory as this notebook)
NOTEBOOK_DIR = Path("/global/homes/d/dkololgi/TNG/Illustris/workflows/abacus_tweb").resolve()
OUT_HTML = NOTEBOOK_DIR / "visualize_staged_mocks_wedge_cweb_3d.html"

missing = [p for _, p in STAGES if not p.exists()]
if missing:
    msg = "Missing staged-mock wedge NPZ(s). Build them first, then re-run:\n"
    msg += "\n".join(f"  - {p}" for p in missing)
    raise FileNotFoundError(msg)

print("Wedge dir:", WEDGE_DIR)
print("lambda_thr:", LAMBDA_THRESHOLD)
print("MAX_POINTS:", MAX_POINTS)
for label, p in STAGES:
    print(f"  OK  {p.name}  ({label})")


In [ ]:
CLASS_NAMES = np.array(["void", "wall", "filament", "cluster"])
COLOR_MAP = {
    0: "#4C78A8",  # blue
    1: "#F58518",  # orange
    2: "#54A24B",  # green
    3: "#E45756",  # red
}


def _pick_array(data, candidates):
    for key in candidates:
        if key in data:
            return np.asarray(data[key])
    return None


def classify_from_lambdas(lam, threshold):
    lam = np.asarray(lam, dtype=np.float64)
    if lam.ndim != 2 or lam.shape[1] != 3:
        raise ValueError(f"Expected (N,3) eigenvalues, got {lam.shape}")
    return (lam > float(threshold)).sum(axis=1).astype(np.int8)


def load_staged_npz(path, lambda_threshold=LAMBDA_THRESHOLD):
    """Load one funnel-stage NPZ; return arrays and full-sample class fractions."""
    data = np.load(path)
    ra = np.asarray(data["ra"], dtype=np.float64)
    dec = np.asarray(data["dec"], dtype=np.float64)
    zz = np.asarray(data["z"], dtype=np.float64)

    cls = _pick_array(data, ("cls", "CLS", "cweb_class", "CWEB_CLASS"))
    if cls is None:
        l1 = _pick_array(data, ("lambda1", "LAMBDA1", "lam1"))
        l2 = _pick_array(data, ("lambda2", "LAMBDA2", "lam2"))
        l3 = _pick_array(data, ("lambda3", "LAMBDA3", "lam3"))
        if l1 is None or l2 is None or l3 is None:
            raise KeyError(
                f"{path.name}: need 'cls' or lambda1/2/3; keys={list(data.files)}"
            )
        lam = np.stack([l1, l2, l3], axis=1)
        cls = classify_from_lambdas(lam, lambda_threshold)
    else:
        cls = np.asarray(cls, dtype=np.int8)

    assert ra.shape == dec.shape == zz.shape == cls.shape, (
        path.name,
        ra.shape,
        dec.shape,
        zz.shape,
        cls.shape,
    )

    fr = np.bincount(cls.astype(np.int64), minlength=4).astype(np.float64)
    fr /= max(1.0, float(cls.size))
    return {
        "path": Path(path),
        "ra": ra,
        "dec": dec,
        "z": zz,
        "cls": cls,
        "fractions": fr,
        "n": int(ra.size),
    }


def sky_to_xyz(ra_deg, dec_deg, z, mpc_h=MPC_H):
    """RA/DEC/Z -> comoving Cartesian Mpc (Planck18)."""
    from astropy.cosmology import Planck18 as cosmo

    ra_rad = np.deg2rad(np.asarray(ra_deg, dtype=np.float64))
    dec_rad = np.deg2rad(np.asarray(dec_deg, dtype=np.float64))
    dist = cosmo.comoving_distance(np.asarray(z, dtype=np.float64)).value
    if mpc_h:
        dist = dist * float(cosmo.h)
    x = dist * np.cos(dec_rad) * np.cos(ra_rad)
    y = dist * np.cos(dec_rad) * np.sin(ra_rad)
    z3 = dist * np.sin(dec_rad)
    return x, y, z3


def subsample_indices(n, max_points, seed):
    idx = np.arange(n, dtype=np.int64)
    if max_points and n > int(max_points):
        rng = np.random.default_rng(int(seed))
        idx = np.sort(rng.choice(idx, size=int(max_points), replace=False))
    return idx


def fraction_annotation(fr, title_prefix=""):
    lines = [f"{CLASS_NAMES[k]}: {fr[k]*100:.2f}%" for k in range(4)]
    body = "<br>".join(lines)
    return f"{title_prefix}{body}" if title_prefix else body


In [ ]:
# Load all funnel stages
loaded = []
for label, npz_path in STAGES:
    rec = load_staged_npz(npz_path)
    rec["label"] = label
    loaded.append(rec)
    print(f"{label}: N={rec['n']:,}  fractions={rec['fractions']}")


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

units = "Mpc/h" if MPC_H else "Mpc"


def make_stage_scatter3d(rec, seed_offset=0, show=True):
    """One Plotly 3D scatter for a funnel stage; returns (fig, plot_fractions)."""
    ra, dec, zz, cls = rec["ra"], rec["dec"], rec["z"], rec["cls"]
    idx = subsample_indices(ra.size, MAX_POINTS, SEED + seed_offset)
    x, y, z3 = sky_to_xyz(ra[idx], dec[idx], zz[idx])
    cls_plot = cls[idx].astype(np.int16)
    colors = np.array([COLOR_MAP[int(c)] for c in cls_plot])
    hover = [
        f"class={CLASS_NAMES[int(c)]}<br>RA={ra[i]:.3f} DEC={dec[i]:.3f} z={zz[i]:.5f}"
        for i, c in zip(idx.tolist(), cls_plot.tolist())
    ]

    fr_plot = np.bincount(cls_plot.astype(np.int64), minlength=4).astype(np.float64)
    fr_plot /= max(1.0, float(cls_plot.size))

    title = (
        f"{rec['label']}"
        f"<br><sup>{rec['path'].name} | N={rec['n']:,} | Nplot={idx.size:,} | "
        f"lambda_thr={LAMBDA_THRESHOLD:g}</sup>"
    )

    fig = go.Figure(
        data=go.Scatter3d(
            x=x,
            y=y,
            z=z3,
            mode="markers",
            marker=dict(size=2, opacity=0.6, color=colors),
            text=hover,
            hoverinfo="text",
        )
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title=f"x [{units}]",
            yaxis_title=f"y [{units}]",
            zaxis_title=f"z [{units}]",
            aspectmode="data",
        ),
        template="plotly_white",
        height=750,
        showlegend=False,
    )
    fig.add_annotation(
        text=fraction_annotation(fr_plot, title_prefix="Plotted subset<br>"),
        xref="paper",
        yref="paper",
        x=0.01,
        y=0.99,
        showarrow=False,
        align="left",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1,
    )
    # Full-wedge fractions (not subsampled)
    fig.add_annotation(
        text="Full wedge<br>" + fraction_annotation(rec["fractions"]),
        xref="paper",
        yref="paper",
        x=0.99,
        y=0.99,
        showarrow=False,
        align="right",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1,
    )
    if show:
        fig.show()
    return fig, fr_plot


# Four figures — one per funnel stage
stage_figs = []
for k, rec in enumerate(loaded):
    fig, _ = make_stage_scatter3d(rec, seed_offset=k, show=True)
    stage_figs.append(fig)


In [ ]:
# Combined 2x2 subplot figure + write HTML
fig_grid = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"type": "scatter3d"}] * 2] * 2,
    subplot_titles=[rec["label"] for rec in loaded],
    horizontal_spacing=0.02,
    vertical_spacing=0.06,
)

for k, rec in enumerate(loaded):
    row, col = divmod(k, 2)
    ra, dec, zz, cls = rec["ra"], rec["dec"], rec["z"], rec["cls"]
    idx = subsample_indices(ra.size, MAX_POINTS, SEED + 100 + k)
    x, y, z3 = sky_to_xyz(ra[idx], dec[idx], zz[idx])
    cls_plot = cls[idx].astype(np.int16)
    colors = [COLOR_MAP[int(c)] for c in cls_plot]

    fig_grid.add_trace(
        go.Scatter3d(
            x=x,
            y=y,
            z=z3,
            mode="markers",
            marker=dict(size=1.5, opacity=0.55, color=colors),
            name=rec["label"],
            showlegend=False,
        ),
        row=row + 1,
        col=col + 1,
    )

# Class-fraction annotations (paper coords, one per panel)
for k, rec in enumerate(loaded):
    row, col = divmod(k, 2)
    x_anchor = 0.02 + col * 0.5
    y_anchor = 0.98 - row * 0.48
    fig_grid.add_annotation(
        text=f"N={rec['n']:,}<br>" + fraction_annotation(rec["fractions"]),
        xref="paper",
        yref="paper",
        x=x_anchor,
        y=y_anchor,
        showarrow=False,
        align="left",
        font=dict(size=9),
        bgcolor="rgba(255,255,255,0.75)",
    )

fig_grid.update_layout(
    title=(
        "Staged mocks wedge cosmic web funnel (truth-only)"
        f"<br><sup>lambda_thr={LAMBDA_THRESHOLD:g} | wedge RA [{RA_MIN},{RA_MAX}] "
        f"Dec [{DEC_MIN},{DEC_MAX}] Z [{Z_MIN},{Z_MAX}] | units={units}</sup>"
    ),
    template="plotly_white",
    height=1100,
    showlegend=False,
)

# Scene axis labels for each 3D subplot
scene_names = ["scene", "scene2", "scene3", "scene4"]
for sn in scene_names:
    if sn in fig_grid.layout:
        fig_grid.layout[sn].update(
            xaxis_title=f"x [{units}]",
            yaxis_title=f"y [{units}]",
            zaxis_title=f"z [{units}]",
            aspectmode="data",
        )

fig_grid.write_html(str(OUT_HTML), include_plotlyjs="cdn")
print("Wrote:", OUT_HTML)
fig_grid.show()


In [ ]:
# Optional: bar chart comparing class fractions across funnel stages
import plotly.express as px
import pandas as pd

rows = []
for rec in loaded:
    for k, name in enumerate(CLASS_NAMES):
        rows.append({
            "stage": rec["label"],
            "class": str(name),
            "fraction": float(rec["fractions"][k]),
        })

df = pd.DataFrame(rows)
fig_bar = px.bar(
    df,
    x="stage",
    y="fraction",
    color="class",
    barmode="group",
    color_discrete_map={
        "void": COLOR_MAP[0],
        "wall": COLOR_MAP[1],
        "filament": COLOR_MAP[2],
        "cluster": COLOR_MAP[3],
    },
    title=f"Cosmic web class fractions by funnel stage (lambda_thr={LAMBDA_THRESHOLD:g})",
)
fig_bar.update_layout(template="plotly_white", height=450, yaxis_tickformat=".1%")
fig_bar.show()
